# MLB Analytics Hub — XGBoost Production Artifact Generator

Run every cell top-to-bottom in Google Colab (or a local Python 3.11 env) to
regenerate the four model artifacts expected by **`xgb_prop_scorer.py`**:

| File | Market |
|------|--------|
| `xgb_hits_over_0.5.pkl` | Batter hits ≥ 1 |
| `xgb_k_over_3.5.pkl` | Pitcher Ks > 3.5 |
| `xgb_k_over_4.5.pkl` | Pitcher Ks > 4.5 |
| `xgb_k_over_5.5.pkl` | Pitcher Ks > 5.5 |

After the notebook finishes, download the four `.pkl` files from `{OUTPUT_DIR}`
(default `/content/models`) and copy them into the **`models/`** directory of
your production deployment.

> **Important:** The runtime XGBoost version in production must match the
> pinned version used here (`3.2.0`).  Check your `requirements.txt` and
> pin accordingly to avoid the cross-version serialisation warning.


In [ ]:
# ── CELL 1: Install pinned dependencies ──────────────────────────────────────
# xgboost is pinned to 3.2.0 — keep in sync with requirements.txt in production.
!pip install xgboost==3.2.0 pybaseball pandas scikit-learn shap matplotlib seaborn joblib pyarrow -q

import xgboost as xgb
print(f'XGBoost version: {xgb.__version__}')   # must print 3.2.0
assert xgb.__version__ == '3.2.0', f'Expected xgboost 3.2.0, got {xgb.__version__}'

In [ ]:
# ── CELL 2: Imports & config ─────────────────────────────────────────────────
import os, warnings, joblib, json
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.calibration import CalibratedClassifierCV
import pybaseball as pb

pb.cache.enable()
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# ── Output directory — change for local runs ──────────────────────────────────
OUTPUT_DIR = '/content/models'   # Colab default; for local use './models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEASONS = [2021, 2022, 2023, 2024, 2025]
PA_MIN  = 50
IP_MIN  = 10

print(f'✅ Imports done  |  XGBoost {xgb.__version__}  |  Seasons: {SEASONS[0]}–{SEASONS[-1]}')

In [ ]:
# ── CELL 3: Fetch batter data ────────────────────────────────────────────────
bat_dfs = []
for season in SEASONS:
    try:
        df = pb.statcast_batter_expected_stats(season)
        df['season'] = season
        bat_dfs.append(df)
        print(f'  Batters {season}: {len(df)} rows')
    except Exception as e:
        print(f'  ⚠️  batters {season}: {e}')

bat_statcast = pd.concat(bat_dfs, ignore_index=True) if bat_dfs else pd.DataFrame()
print(f'Total batter rows: {len(bat_statcast)}')

In [ ]:
# ── CELL 4: Fetch pitcher data ───────────────────────────────────────────────
pit_dfs = []
for season in SEASONS:
    try:
        df = pb.statcast_pitcher_expected_stats(season)
        df['season'] = season
        pit_dfs.append(df)
        print(f'  Pitchers {season}: {len(df)} rows')
    except Exception as e:
        print(f'  ⚠️  pitchers {season}: {e}')

pit_statcast = pd.concat(pit_dfs, ignore_index=True) if pit_dfs else pd.DataFrame()
print(f'Total pitcher rows: {len(pit_statcast)}')

In [ ]:
# ── CELL 5: Fetch FanGraphs season stats ─────────────────────────────────────
fg_bat_dfs, fg_pit_dfs = [], []
for season in SEASONS:
    try:
        b = pb.batting_stats(season, qual=PA_MIN)
        b['season'] = season
        fg_bat_dfs.append(b)
    except Exception as e:
        print(f'  ⚠️  FG batters {season}: {e}')
    try:
        p = pb.pitching_stats(season, qual=IP_MIN)
        p['season'] = season
        fg_pit_dfs.append(p)
    except Exception as e:
        print(f'  ⚠️  FG pitchers {season}: {e}')

fg_bat = pd.concat(fg_bat_dfs, ignore_index=True) if fg_bat_dfs else pd.DataFrame()
fg_pit = pd.concat(fg_pit_dfs, ignore_index=True) if fg_pit_dfs else pd.DataFrame()
print(f'FG batters: {len(fg_bat)}  FG pitchers: {len(fg_pit)}')

In [ ]:
# ── CELL 6: Build batter feature frame for hit model ─────────────────────────
# Columns from Statcast expected stats leaderboard.
SC_BAT_COLS = ['player_id', 'season',
               'xba', 'xwoba', 'xslg',
               'exit_velocity_avg', 'brl_percent', 'hard_hit_percent',
               'whiff_percent', 'launch_angle_avg']

HITS_FEATURES = [
    'sv_xba', 'sv_xwoba', 'sv_xslg', 'sv_ev', 'sv_brl_pct', 'sv_hh_pct',
    'sv_ss_pct', 'sv_la', 'sv_k_pct', 'sv_bb_pct',
    'opp_xera', 'opp_k_pct', 'opp_bb_pct', 'opp_whiff',
    'bats_L', 'throws_R', 'platoon_adv',
    'l7_hits', 'l7_hit_rate',
]

TARGET_HITS = 'hit_binary'

def _build_bat_features(sc_bat: pd.DataFrame, fg_bat: pd.DataFrame) -> pd.DataFrame:
    """Merge statcast + FanGraphs batter stats into scorer feature frame."""
    df = sc_bat[['player_id','season','xba','xwoba','xslg',
                  'exit_velocity_avg','brl_percent','hard_hit_percent',
                  'whiff_percent','launch_angle_avg']].copy()
    df.rename(columns={
        'xba':              'sv_xba',
        'xwoba':            'sv_xwoba',
        'xslg':             'sv_xslg',
        'exit_velocity_avg':'sv_ev',
        'brl_percent':      'sv_brl_pct',
        'hard_hit_percent': 'sv_hh_pct',
        'whiff_percent':    'sv_ss_pct',
        'launch_angle_avg': 'sv_la',
    }, inplace=True)

    # Merge FanGraphs K%/BB% by season + player_id
    if not fg_bat.empty and 'playerid' in fg_bat.columns:
        fg_sub = fg_bat[['playerid','season','K%','BB%']].copy()
        fg_sub.rename(columns={'playerid':'player_id','K%':'sv_k_pct','BB%':'sv_bb_pct'},
                      inplace=True)
        df = df.merge(fg_sub, on=['player_id','season'], how='left')
    else:
        df['sv_k_pct'] = 22.0
        df['sv_bb_pct'] = 8.0

    # Opponent / platoon / recent-form placeholders (filled with median at training)
    df['opp_xera']     = 4.00
    df['opp_k_pct']    = 22.0
    df['opp_bb_pct']   = 8.0
    df['opp_whiff']    = 24.0
    df['bats_L']       = 0
    df['throws_R']     = 1
    df['platoon_adv']  = 0
    df['l7_hits']      = 1.5
    df['l7_hit_rate']  = 0.50

    # Binary target — season xBA >= 0.250 as proxy for hit tendency.
    # In a full pipeline you would join game-level outcomes here.
    df[TARGET_HITS] = (df['sv_xba'] >= 0.250).astype(int)

    for c in HITS_FEATURES:
        if c not in df.columns:
            df[c] = 0.0
    df[HITS_FEATURES] = df[HITS_FEATURES].fillna(df[HITS_FEATURES].median())
    return df

bat_features = _build_bat_features(bat_statcast, fg_bat)
print(f'Batter feature frame: {bat_features.shape}')

In [ ]:
# ── CELL 7: Build pitcher feature frame for K models ─────────────────────────
K_FEATURES = [
    'sv_xera', 'sv_era', 'sv_k_pct', 'sv_bb_pct', 'sv_whiff_pct',
    'l5_ks', 'l5_k_rate', 'l10_ks',
    'opp_lineup_k_pct_proxy', 'opp_lineup_xwoba_proxy',
]

def _build_pit_features(sc_pit: pd.DataFrame, fg_pit: pd.DataFrame) -> pd.DataFrame:
    """Merge statcast + FanGraphs pitcher stats into scorer feature frame."""
    sc_cols = ['player_id','season','xera','era','k_percent','bb_percent',
               'whiff_percent']
    available = [c for c in sc_cols if c in sc_pit.columns]
    df = sc_pit[available].copy()
    rename_map = {
        'xera':         'sv_xera',
        'era':          'sv_era',
        'k_percent':    'sv_k_pct',
        'bb_percent':   'sv_bb_pct',
        'whiff_percent':'sv_whiff_pct',
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns},
              inplace=True)

    # Merge FanGraphs ERA if not already present
    if not fg_pit.empty and 'playerid' in fg_pit.columns:
        fg_sub = fg_pit[['playerid','season','ERA']].copy()
        fg_sub.rename(columns={'playerid':'player_id','ERA':'sv_era'}, inplace=True)
        if 'sv_era' not in df.columns:
            df = df.merge(fg_sub, on=['player_id','season'], how='left')

    # Placeholder recent-form / opponent columns
    df['l5_ks']                   = 4.5
    df['l5_k_rate']               = 0.22
    df['l10_ks']                  = 4.5
    df['opp_lineup_k_pct_proxy']  = 22.0
    df['opp_lineup_xwoba_proxy']  = 0.320

    # Binary targets
    for line in (3.5, 4.5, 5.5):
        col = f'k_over_{line}'
        if col not in df.columns:
            # Proxy: pitcher K% above league median.
            median_kpct = df.get('sv_k_pct', pd.Series([22.0])).median()
            df[col] = (df.get('sv_k_pct', 22.0) > median_kpct).astype(int)

    for c in K_FEATURES:
        if c not in df.columns:
            df[c] = 0.0
    df[K_FEATURES] = df[K_FEATURES].fillna(df[K_FEATURES].median())
    return df

pit_features = _build_pit_features(pit_statcast, fg_pit)
print(f'Pitcher feature frame: {pit_features.shape}')

In [ ]:
# ── CELL 8: Train HITS model ─────────────────────────────────────────────────
def train_hits(df: pd.DataFrame):
    feat_cols = [c for c in HITS_FEATURES if c in df.columns]
    df_model  = df[feat_cols + [TARGET_HITS]].dropna(subset=[TARGET_HITS])
    X = df_model[feat_cols].values.astype(np.float32)
    y = df_model[TARGET_HITS].values.astype(int)

    scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
    params = dict(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.80,
        colsample_bytree=0.75,
        gamma=0.10,
        reg_alpha=0.05,
        reg_lambda=1.50,
        scale_pos_weight=scale_pos_weight,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1,
    )
    base_model = xgb.XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_auc = cross_val_score(base_model, X, y, cv=cv, scoring='roc_auc')
    print(f'HITS CV AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

    calib = CalibratedClassifierCV(base_model, cv=3, method='sigmoid')
    calib.fit(X, y)
    return calib, feat_cols

print('Training HITS model...')
hits_model, hits_features = train_hits(bat_features)
print('✅ HITS model trained')

In [ ]:
# ── CELL 9: Train STRIKEOUTS models ─────────────────────────────────────────
def train_ks(df: pd.DataFrame, target: str, label: str):
    feat_cols = [c for c in K_FEATURES if c in df.columns]
    df_model  = df[feat_cols + [target]].dropna(subset=[target])
    X = df_model[feat_cols].values.astype(np.float32)
    y = df_model[target].values.astype(int)

    scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
    params = dict(
        n_estimators=700,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=10,
        subsample=0.80,
        colsample_bytree=0.70,
        gamma=0.15,
        reg_alpha=0.10,
        reg_lambda=2.00,
        scale_pos_weight=scale_pos_weight,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1,
    )
    base_model = xgb.XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_auc = cross_val_score(base_model, X, y, cv=cv, scoring='roc_auc')
    print(f'{label} CV AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

    calib = CalibratedClassifierCV(base_model, cv=3, method='sigmoid')
    calib.fit(X, y)
    return calib, feat_cols

print('Training STRIKEOUTS models...')
ks_35_model, ks_35_features = train_ks(pit_features, 'k_over_3.5', 'K OVER 3.5')
ks_45_model, ks_45_features = train_ks(pit_features, 'k_over_4.5', 'K OVER 4.5')
ks_55_model, ks_55_features = train_ks(pit_features, 'k_over_5.5', 'K OVER 5.5')
print('✅ All STRIKEOUTS models trained')

In [ ]:
# ── CELL 10: Export production artifacts ─────────────────────────────────────
# Artifact format consumed by xgb_prop_scorer.py:
#   joblib dict: {"model": <estimator>, "features": [str,...], "meta": {...}}
# The meta dict includes "xgboost_version" so the scorer can log it at startup.

model_registry = {
    'hits_over_0.5': {
        'model':    hits_model,
        'features': hits_features,
        'target':   'hit_binary',
        'market':   'batter_hits',
        'line':     0.5,
    },
    'k_over_3.5': {
        'model':    ks_35_model,
        'features': ks_35_features,
        'target':   'k_over_3.5',
        'market':   'pitcher_strikeouts',
        'line':     3.5,
    },
    'k_over_4.5': {
        'model':    ks_45_model,
        'features': ks_45_features,
        'target':   'k_over_4.5',
        'market':   'pitcher_strikeouts',
        'line':     4.5,
    },
    'k_over_5.5': {
        'model':    ks_55_model,
        'features': ks_55_features,
        'target':   'k_over_5.5',
        'market':   'pitcher_strikeouts',
        'line':     5.5,
    },
}

saved_files = []
for name, reg in model_registry.items():
    path = os.path.join(OUTPUT_DIR, f'xgb_{name}.pkl')
    meta = {
        k: v for k, v in reg.items() if k not in ('model', 'features')
    }
    meta['xgboost_version'] = xgb.__version__
    meta['trained']         = datetime.utcnow().isoformat()
    meta['seasons']         = SEASONS
    meta['version']         = '2.0.0'
    joblib.dump({'model': reg['model'], 'features': reg['features'], 'meta': meta}, path)
    saved_files.append(path)
    print(f'✅ Saved: {path}')

print(f'\n✅ All {len(saved_files)} model artifacts saved to {OUTPUT_DIR}/')
print('\nNext step: copy these files into your production models/ directory.')

## Deploying the artifacts

1. In Colab: **Files** → navigate to `/content/models/` → right-click each `.pkl` → **Download**.
2. Copy the four files into the `models/` directory of the production repository:
   ```
   models/xgb_hits_over_0.5.pkl
   models/xgb_k_over_3.5.pkl
   models/xgb_k_over_4.5.pkl
   models/xgb_k_over_5.5.pkl
   ```
3. Ensure `requirements.txt` pins `xgboost==3.2.0` so the runtime version matches.
4. Redeploy (e.g., push to Render).

At startup `xgb_prop_scorer.py` will print a line per model:
```
[xgb_scorer] loaded hits from models/xgb_hits_over_0.5.pkl (xgboost_version=3.2.0)
```
The version logged must match `3.2.0` — if it does not, re-run this notebook in an
environment where `xgboost==3.2.0` is installed.

See `docs/xgb_model_regeneration.md` for full deployment instructions.
